In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('/content/final_dataset.csv')

In [3]:
df

,label,text
0,0,annual physical scheduled 5 jun 2025 1015 loca...
1,0,please save e mail records dear craig congratu...
2,1,need urgent assistance donald williams esq 60 ...
3,2,megasize unit megadlk huge advancement mens he...
4,0,original message rodenburg eric enron sent mon...
...,...,...
514613,1,remote customer service rep needed 800week int...
514614,0,dear businessweek online user outlook widely c...
514615,0,next scheduled rcr meeting take place follows ...
514616,1,work home earn 1500week liking facebook posts ...


In [4]:
df.shape

(514618, 2)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514618 entries, 0 to 514617
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   514618 non-null  int64 
 1   text    514618 non-null  object
dtypes: int64(1), object(1)
memory usage: 7.9+ MB


In [6]:
df.isnull().sum()

,0
label,0
text,0


In [7]:
df.describe()

,label
count,514618.000000
mean,0.822233
std,0.692418
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,2.000000


In [8]:
df.columns

Index(['label', 'text'], dtype='object')

In [9]:
df["label"].value_counts()

,count
label,
1,251626
0,177237
2,85755


In [10]:
for label in sorted(df["label"].unique()):
    print(f"\n===== Label {label} =====")
    print(df[df["label"] == label]["text"].head(5).to_string(index=False))


===== Label 0 =====
annual physical scheduled 5 jun 2025 1015 locat...
please save e mail records dear craig congratul...
original message rodenburg eric enron sent mond...
wondered laptops found work well dmdx inspiron ...
sbi account xxxx1234 credited rs 25500 17 jul 2025

===== Label 1 =====
need urgent assistance donald williams esq 60 c...
flipkart account lucky prize rs 48500 claim wit...
microsoft support detected malware pc call toll...
dhl package held customs pay clearance fee 1300...
angelina rogers albumena716westendcitycentercom...

===== Label 2 =====
megasize unit megadlk huge advancement mens hea...
                          finntresa demetrissarita
make three canings flogging sock filled manure ...
escapenumber hours needs http jfwp escapelong b...
cla lis ftb disolv hlf tab toung 9 mins bfore i...


In [11]:
df.groupby("label").head(5)

,label,text
0,0,annual physical scheduled 5 jun 2025 1015 loca...
1,0,please save e mail records dear craig congratu...
2,1,need urgent assistance donald williams esq 60 ...
3,2,megasize unit megadlk huge advancement mens he...
4,0,original message rodenburg eric enron sent mon...
5,0,wondered laptops found work well dmdx inspiron...
6,0,sbi account xxxx1234 credited rs 25500 17 jul ...
8,1,flipkart account lucky prize rs 48500 claim wi...
10,2,finntresa demetrissarita
13,2,make three canings flogging sock filled manure...


In [12]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [13]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"])

In [14]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [15]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128)

In [16]:
import torch

class SMSDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [17]:
train_dataset = SMSDataset(train_encodings, train_labels)
test_dataset = SMSDataset(test_encodings, test_labels)

In [18]:
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

Training samples: 411694
Testing samples: 102924


In [19]:
from transformers import BertForSequenceClassification

In [20]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=64)

In [22]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(),lr=2e-5)

In [24]:
print(len(train_dataset))

411694


In [25]:
print(len(train_loader))

25731


In [26]:
print(train_loader.batch_size)

16


In [28]:
print(len(df))

514618


In [29]:
print(len(train_texts))
print(len(test_texts))

411694
102924


In [30]:
print(len(train_labels))
print(len(test_labels))

411694
102924


In [32]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514618 entries, 0 to 514617
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   514618 non-null  int64 
 1   text    514618 non-null  object
dtypes: int64(1), object(1)
memory usage: 7.9+ MB
None


In [23]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score

epochs = 3

train_losses = []
eval_losses = []

train_accs = []
eval_accs = []

for epoch in range(epochs):

    # -------------------
    # Training
    # -------------------
    model.train()

    total_train_loss = 0
    train_preds = []
    train_labels_list = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in loop:

        optimizer.zero_grad()

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        total_train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels_list.extend(labels.cpu().numpy())

        loss.backward()
        optimizer.step()

    avg_train_loss = total_train_loss / len(train_loader)
    train_acc = accuracy_score(train_labels_list, train_preds)

    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)



Epoch 1:   2%|▏         | 503/25731 [25:50<21:07:41,  3.01s/it]Exception ignored in: <generator object tqdm.__iter__ at 0x781b16cd7880>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1196, in __iter__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1265, in close
    def close(self):

KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
# -------------------
    # Evaluation
    # -------------------
    model.eval()

    total_eval_loss = 0
    eval_preds = []
    eval_labels_list = []

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_eval_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            eval_preds.extend(preds.cpu().numpy())
            eval_labels_list.extend(labels.cpu().numpy())

    avg_eval_loss = total_eval_loss / len(test_loader)
    eval_acc = accuracy_score(eval_labels_list, eval_preds)

    eval_losses.append(avg_eval_loss)
    eval_accs.append(eval_acc)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss : {avg_train_loss:.4f}")
    print(f"Test Loss  : {avg_eval_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Test Acc   : {eval_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(train_losses, marker='o', label='Train Loss')
plt.plot(eval_losses, marker='o', label='Test Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BERT Loss over Epochs')

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(train_accs, marker='o', label='Train Accuracy')
plt.plot(eval_accs, marker='o', label='Test Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('BERT Accuracy over Epochs')

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
avg_train_loss = total_loss / len(train_loader)
train_losses.append(avg_train_loss)

# avg_val_loss = total_val_loss / len(val_loader)
# val_losses.append(avg_val_loss)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report)

import numpy as np
import torch

model.eval()

predictions = []
true_labels = []

In [ ]:
with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(input_ids=input_ids,attention_mask=attention_mask)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

In [ ]:
accuracy = accuracy_score(true_labels, predictions)

precision = precision_score(
    true_labels,
    predictions,
    average="weighted")

recall = recall_score(
    true_labels,
    predictions,
    average="weighted")

f1 = f1_score(
    true_labels,
    predictions,
    average="weighted")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

In [ ]:
print(classification_report(true_labels,predictions,target_names=["Ham", "Spam", "Phishing"]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(true_labels, predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Ham", "Spam", "Phishing"])

disp.plot(cmap="Blues")
plt.title("Confusion Matrix - BERT")
plt.show()

In [ ]:
results = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1}

print(results)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english'
)

X_train = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, train_labels)

In [ ]:
lr_predictions = lr_model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

lr_accuracy = accuracy_score(test_labels, lr_predictions)
lr_precision = precision_score(test_labels, lr_predictions, average="weighted")
lr_recall = recall_score(test_labels, lr_predictions, average="weighted")
lr_f1 = f1_score(test_labels, lr_predictions, average="weighted")

print("Logistic Regression")
print("Accuracy :", lr_accuracy)
print("Precision:", lr_precision)
print("Recall   :", lr_recall)
print("F1-score :", lr_f1)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train, train_labels)

In [ ]:
nb_predictions = nb_model.predict(X_test)

In [ ]:
nb_accuracy = accuracy_score(test_labels, nb_predictions)
nb_precision = precision_score(test_labels, nb_predictions, average="weighted")
nb_recall = recall_score(test_labels, nb_predictions, average="weighted")
nb_f1 = f1_score(test_labels, nb_predictions, average="weighted")

print("Naive Bayes")
print("Accuracy :", nb_accuracy)
print("Precision:", nb_precision)
print("Recall   :", nb_recall)
print("F1-score :", nb_f1)

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "BERT"],
    "Accuracy": [
        nb_accuracy,
        lr_accuracy,
        accuracy
    ],
    "Precision": [
        nb_precision,
        lr_precision,
        precision
    ],
    "Recall": [
        nb_recall,
        lr_recall,
        recall
    ],
    "F1-score": [
        nb_f1,
        lr_f1,
        f1
    ]
})

comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.set_index("Model")[["Accuracy", "F1-score"]].plot(kind="bar")

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()